# Pipeline: differential compartments

Part of the **[Fig. 4 chapter](fig4.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}L1color.tsv'`  ·  _metadata: color_
- `f'{compdir}L1/comp.impute.hdf'`  ·  _other_
- `f'{compdir}L1/comp.{mode}.hdf'`  ·  _other_
- `f'{outdir}DifferentialResult/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph'`  ·  _other_
- `f'{outdir}bin_stats.hdf'`  ·  _other_
- `f'{indir}analysis/mCG_distribution/L1_chrom100k_mCG.hdf'`  ·  _other_
- `f'{indir}analysis/mCH_distribution/L1_chrom100k_mCH.hdf'`  ·  _other_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [2]:
import os
import cooler
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib import cm as cm
import seaborn as sns

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [3]:
group_name = 'all'

In [4]:
indir = f'{ENTEX_ROOT}/'
compdir = f'{indir}analysis/compartment/'
outdir = f'{indir}analysis/diff_comp/{group_name}/'


In [5]:
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35', 'c36'], axis=0)
L1_annot = L1_meta['L1_abbr'].to_dict()
L1_color = L1_meta['color'].to_dict()
res = 100000
L1_color.update({L1_annot[k]: L1_color[k] for k in L1_annot if k in L1_color})  # also key by name


In [6]:
comp = pd.read_hdf(f'{compdir}L1/comp.impute.hdf')
comp

In [7]:
mode = 'impute'
comp = pd.read_hdf(f'{compdir}L1/comp.{mode}.hdf', key='data')
comp.index = comp['chrom'].astype(str) + '-' + (comp['start'] // res).astype(str)
comp = comp.loc[comp['raw_binfilter']]
print(comp.shape)


In [8]:
binall = pd.DataFrame(index=comp.index)
binall['chrom'] = binall.index.str.split('-').str[0]
binall['start'] = binall.index.str.split('-').str[1].astype(int) * res
binall['end'] = binall['start'] + res
binall

In [9]:
chrom_size_path = f'{REF_ROOT}/hg38/fasta/hg38.main.chrom.sizes'
chrom_sizes = cooler.read_chromsizes(chrom_size_path, all_names=True)
chrom_sizes = chrom_sizes.iloc[:22]


In [10]:
for xx in L1_meta.index:
    os.makedirs(f'{outdir}{xx}_100Kb_pca/intra_pca/{xx}_100Kb_mat/', exist_ok=True)
    tmp = binall.copy()
    tmp['pc'] = comp[xx]
    for c in chrom_sizes.index:
        tmp.loc[tmp['chrom']==c].to_csv(f'{outdir}{xx}_100Kb_pca/intra_pca/{xx}_100Kb_mat/{c}.pc.bedGraph', sep='\t', header=False, index=False)


In [11]:
tmp = pd.DataFrame(index=L1_meta.index)
tmp['matrix_path'] = '.'
tmp['bed_path'] = '.'
tmp['sample'] = tmp.index + '_100Kb'
tmp['group'] = tmp.index
tmp.to_csv(f'{outdir}input.txt', sep='\t', header=False, index=False)


In [13]:
comp = pd.read_csv(f'{outdir}DifferentialResult/fdr_result/differential.intra_sample_combined.pcQnm.bedGraph', sep='\t', header=0, index_col=None)
comp.index = comp['chr'] + '_' + (comp['start'] // res).astype(str)
comp


In [14]:
binall = comp[['chr', 'start', 'end', 'sample_maha', 'pval', 'padj']]
comp = comp[L1_meta.index]


In [15]:
binall.to_hdf(f'{outdir}bin_stats.hdf', key='data')


In [16]:
binall = pd.read_hdf(f'{outdir}bin_stats.hdf', key='data')


In [17]:
mcg = pd.read_hdf(f'{indir}analysis/mCG_distribution/L1_chrom100k_mCG.hdf', key='data').T
mch = pd.read_hdf(f'{indir}analysis/mCH_distribution/L1_chrom100k_mCH.hdf', key='data').T


In [18]:
selb = comp.index.intersection(mcg.index).intersection(mch.index)
print(len(selb))

In [19]:
comp = comp.loc[selb]
mcg = mcg.loc[selb]
mch = mch.loc[selb]
binall = binall.loc[selb]


In [20]:
from scipy.stats import zscore, pearsonr, norm

# selb = (binall['padj']<1e-3)
selb = zscore(binall['sample_maha'])>norm.isf(0.1)
print(selb.sum())


In [21]:
tmp3c = comp.loc[selb].values
# tmp3c = zscore(tmp3c, axis=1)


In [22]:
rorder = cg.dendrogram_row.reordered_ind.copy()
corder = cg.dendrogram_col.reordered_ind.copy()

In [24]:
def compute_pearsonr(data_x, data_y):
    x_mean = np.mean(data_x, axis=1)
    y_mean = np.mean(data_y, axis=1)
    cov = np.sum((data_x - x_mean[:, None]) * (data_y - y_mean[:, None]), axis=1)
    stdlabel = np.sqrt(np.sum((data_x - x_mean[:, None]) ** 2, axis=1))
    stdpred = np.sqrt(np.sum((data_y - y_mean[:, None]) ** 2, axis=1))
    corr = cov / stdlabel / stdpred
    print(np.mean(corr))
    return corr


In [25]:
binall['mCG_corr'] = compute_pearsonr(comp.values, mcg.values)
binall['mCH_corr'] = compute_pearsonr(comp.values, mch.values)


In [26]:
binall['logPadj'] = -np.log10(binall['padj'])
binall.loc[binall['logPadj']>300, 'logPadj'] = 300

In [27]:
binall['mCG_std'] = np.std(mcg, axis=1)
binall['mCH_std'] = np.std(mch, axis=1)
binall['comp_std'] = np.std(comp, axis=1)
binall

In [28]:
binall.to_hdf(f'{outdir}bin_stats.hdf', key='data')


In [29]:
print(np.sum(zscore(binall['sample_maha'])>norm.isf(0.1)))
